<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline - Random Forest (Compatible NumPy 2.0)
**Proyecto de Maestría:** Detección de Deslizamientos | Universidad EAFIT

Este notebook utiliza Random Forest y HOG para establecer una base de comparación. Corregido para compatibilidad con las últimas librerías de Python.

## 1. Configuración de Acceso a Datos

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Montar Drive
drive.mount('/content/drive')

# Localizar datos
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Datos listos: {len(img_list)} muestras.")
else:
    print("❌ ERROR: No se encontró la ruta.")

## 2. Extracción de Características (HOG + Slope)
Combinamos la textura visual (HOG) con la información del terreno (Pendiente).

In [ ]:
N_SAMPLES = 1000 
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Procesando"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # --- CORRECCIÓN NUMPY 2.0 ---
    # Extraer RGB y normalizar usando np.ptp() en lugar de .ptp()
    rgb = patch[:,:,[3,2,1]]
    r_min = rgb.min()
    r_range = np.ptp(rgb)
    rgb_norm = ((rgb - r_min) / (r_range + 1e-8) * 255).astype("uint8")
    
    # 1. Características de Textura (HOG)
    feats_hog = hog(rgb_norm, pixels_per_cell=(16,16), cells_per_block=(2,2), 
                channel_axis=-1, feature_vector=True)
    
    # 2. Características Geotécnicas (Media de la Pendiente)
    # La banda 13 (índice 12) es Slope
    avg_slope = np.mean(patch[:,:,12])
    
    # Combinar características
    final_features = np.append(feats_hog, avg_slope)
    
    X.append(final_features)
    y.append(int(mask.max() > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset preparado con {X.shape[1]} variables.")

## 3. Entrenamiento y Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    model.fit(X[t_idx], y[t_idx])
    
    preds = model.predict(X[v_idx])
    f1 = f1_score(y[v_idx], preds)
    scores.append(f1)
    print(f"Fold {fold+1} F1-Score: {f1:.4f}")

print(f"\n🚀 BASELINE FINAL: {np.mean(scores):.4f} ± {np.std(scores):.4f}")